# Phase 1: Exploratory Data Analysis

In this notebook, we'll load the ISOT Fake News dataset, explore its properties, and perform basic text analysis.

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context('notebook')
os.makedirs('../outputs', exist_ok=True)

## 1. Load Data

In [ ]:
fake_path = '../data/Fake.csv'
true_path = '../data/True.csv'

if not os.path.exists(fake_path) or not os.path.exists(true_path):
    print("Datasets not found in ../data/ directory. Please download from Kaggle.")
    print("Creating dummy data for demonstration...")
    df_fake = pd.DataFrame({'title': ['Fake Title 1', 'Fake Title 2'], 'text': ['Fake text 1', 'Fake text 2']})
    df_true = pd.DataFrame({'title': ['Real Title 1', 'Real Title 2'], 'text': ['Real text 1', 'Real text 2']})
else:
    df_fake = pd.read_csv(fake_path)
    df_true = pd.read_csv(true_path)

df_fake['label'] = 1
df_true['label'] = 0

df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df['full_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
print(f"Total shape: {df.shape}")

## 2. Basic Statistics

In [ ]:
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())
print("\nDuplicates:")
print(df.duplicated().sum())

## 3. Class Distribution

In [ ]:
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='label', data=df, palette=['#11998e', '#ff416c'])
plt.title('Class Distribution (0: Real, 1: Fake)')
plt.xlabel('Label')
plt.ylabel('Count')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + 0.35, p.get_height() + 100))

plt.savefig('../outputs/class_distribution.png', dpi=150)
plt.show()

## 4. Article Length Analysis

In [ ]:
df['word_count'] = df['full_text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
sns.histplot(data=df[df['word_count'] < 2000], x='word_count', hue='label', bins=50, kde=True, palette=['#11998e', '#ff416c'])
plt.title('Article Length Distribution (Words)')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.savefig('../outputs/article_length_distribution.png', dpi=150)
plt.show()

## 5. Sample Articles

In [ ]:
print("--- Fake News Samples ---")
for idx, text in enumerate(df[df['label'] == 1]['full_text'].head(3)):
    print(f"[{idx+1}] {text[:200]}...\n")

print("\n--- Real News Samples ---")
for idx, text in enumerate(df[df['label'] == 0]['full_text'].head(3)):
    print(f"[{idx+1}] {text[:200]}...\n")

## 6. Save Combined Dataset

In [ ]:
df.to_csv('../outputs/combined_news.csv', index=False)
print("Dataset saved to ../outputs/combined_news.csv")